# Pipeline 3: The Video Expert

In [20]:
from IPython.display import HTML # to display videos
import base64 # to encode videos as base64
from base64 import b64encode # to encode videos as base64
import os # to interact with the operating system
import subprocess # to run commands
import time # to measure execution time
import csv # to save comments
import uuid # to generate unique ids
import cv2 # to split videos
from PIL import Image # to display videos
import pandas as pd # to display comments
import numpy as np # to use Numerical Python
from io import BytesIO #for a binary stream of data in memory

In [21]:
def download(directory, filename):
    base_url = 'https://raw.githubusercontent.com/Denis2054/RAG-Driven-Generative-AI/main/'

    file_url = f"{base_url}{directory}/{filename}"

    try:

        curl_command = f'curl -o {filename} {file_url}'

        subprocess.run(curl_command, check=True, shell=True)
        print(f"Downloaded '{filename}' successfully.")
    except subprocess.CalledProcessError:
        print(f"Failed to download '{filename}'. Check the URL, your internet connection and the file path")

In [22]:
import os
from dotenv import load_dotenv
load_dotenv()
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_environment = os.getenv("PINECONE_ENVIRONMENT")

if not pinecone_api_key or not pinecone_environment:
    raise ValueError("Pinecone API key and environment must be set in the .env file.")
else:
    print("Pinecone API key and environment loaded successfully.")

Pinecone API key and environment loaded successfully.


In [23]:
from pinecone import ServerlessSpec

index_name = 'videos-sports-us'
cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

In [24]:
import os
from pinecone import Pinecone, ServerlessSpec

api_key = os.environ.get('PINECONE_API_KEY') or 'PINECONE_API_KEY'

from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=api_key)

In [25]:
index = pc.Index(index_name)
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

### Defining the RAG functions

In [26]:
import time 
from sentence_transformers import SentenceTransformer

embedding_model = "all-MiniLM-L6-v2"


def create_embeddings(chunks, embedding_model):
    model = SentenceTransformer(embedding_model)
    start_time = time.time()
    embeddings = model.encode(chunks, show_progress_bar=True)
    end_time = time.time()
    return embeddings

In [36]:
def query_pinecone(query_text, k):
    query_embedding = create_embeddings([query_text], embedding_model)[0]
    query_embedding = query_embedding.tolist()
    query_results = index.query(vector=query_embedding, top_k=k, include_metadata=True)
    return query_results

In [29]:
def collect_query_results(query_results):
    results = []
    for match in query_results['matches']:
        # Prepare the result dictionary for each match
        result = {
            "ID": match['id'],
            "Score": match['score']
        }

        # Check if metadata is available and add to result dictionary
        if 'metadata' in match:
            metadata = match['metadata']
            result['Text'] = metadata.get('text', "No text metadata available.")
            result['Frame Number'] = metadata.get('frame_number', "No frame number available.")
            result['File Name'] = metadata.get('file_name', "No file name available.")
        else:
            result['Text'] = "No metadata available."
            result['Frame Number'] = "No metadata available."
            result['File Name'] = "No metadata available."

        results.append(result)

    return results

In [30]:
import groq
from groq import Groq


def get_response(prompt):
    client = Groq()
    response = client.generate(
        model="llama3-8b-instruct",
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=200,
        temperature=0.7
    )
    return response


### Display video

In [31]:
def display_video(file_name):
  with open(file_name, 'rb') as file:
      video_data = file.read()

  video_url = b64encode(video_data).decode()

  html = f'''
  <video width="640" height="480" controls>
    <source src="data:video/mp4;base64,{video_url}" type="video/mp4">
  Your browser does not support the video tag.
  </video>
  '''
  HTML(html)
  return HTML(html)

In [32]:
def display_video_frame(file_name, frame_number=0, size=(100, 110)):
    cap = cv2.VideoCapture(file_name)

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

    success, frame = cap.read()
    if not success:
        return "Failed to grab frame"

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    img = Image.fromarray(frame)
    img = img.resize(size, Image.ANTIALIAS)  # Resize image to specified size

    buffered = BytesIO()
    img.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode()

    html_str = f'''
    <img src="data:image/jpeg;base64,{img_str}" width="{size[0]}" height="{size[1]}">
    '''
    display(HTML(html_str))
    return HTML(html_str)


In [33]:
import os
from IPython.display import Image, display

def display_frame(frame):
    directory = '/content/'  # Adjust the directory if needed
    file_path = os.path.join(directory, frame)

    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path)
        print(f"File '{frame}' exists. Size: {file_size} bytes.")

        logical_size = 1000 
        if file_size > logical_size:
            print("The file size is greater than the logical value.")
            display(Image(filename=file_path))
        else:
            print("The file size is less than or equal to the logical value.")
    else:
        print(f"File '{frame}' does not exist in the specified directory.")

### Querying the vector store


In [34]:
k = 1

In [37]:
query_text = "Find a basketball player that is scoring with a dunk."
query_results = query_pinecone(query_text,k)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Fri, 17 Apr 2026 11:38:51 GMT', 'Content-Type': 'application/json', 'Content-Length': '103', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '45', 'x-envoy-upstream-service-time': '46', 'x-pinecone-response-duration-ms': '48', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Vector dimension 384 does not match the dimension of the index 1536","details":[]}


In [ ]:
collected_results = collect_query_results(query_results)

for result in collected_results:
  id= result['ID']
  score= result['Score']
  text= result['Text']
  frame= result['Frame Number']
  file_name= result['File Name']
  print(f"ID={id}")
  print(f"score={score}")
  print(f"text={text}")
  print(f"frame_number={frame}")
  print(f"file_name={file_name}")
  print()  # Add a newline for better readability between entries

In [ ]:
directory = "Chapter10/videos"
download(directory,file_name)
display_video(file_name)

In [ ]:
file_name_root = file_name.split('.')[0]
frame="frame_"+str(frame)+".jpg"
print(frame)
directory = "Chapter10/frames/"+file_name_root
print(directory)
download(directory,frame)
display_frame(frame)

### Retrieval Augmented Generation

In [ ]:
prompt=text

In [ ]:
response_content = get_response(prompt)
print(response_content)

# Evaluator

In [ ]:
!pip install spacy

In [ ]:
!python -m spacy download en_core_web_md

In [ ]:
!pip install sentence-transformers==3.0.1

In [ ]:
from sentence_transformers import SentenceTransformer
from torch import cosine_similarity
model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_cosine_similarity_with_embeddings(text1, text2):
    embeddings1 = model.encode(text1)
    embeddings2 = model.encode(text2)
    similarity = cosine_similarity([embeddings1], [embeddings2])
    return similarity[0][0]

In [ ]:
import spacy

def spacy_similarity(text1, text2):
    nlp = spacy.load('en_core_web_md')

    doc1 = nlp(text1)
    doc2 = nlp(text2)

    similarity_score = doc1.similarity(doc2)

    return similarity_score

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(text1, text2):
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([text1, text2])
    similarity = cosine_similarity(tfidf[0:1], tfidf[1:2])
    return similarity[0][0]

In [ ]:
text1 = " In this image, a basketball player is captured mid-air, executing a powerful dunk. With one arm extended towards the hoop and the basketball firmly in hand, the athlete is poised to slam the ball through the net. The player's athletic wear and basketball shoes highlight their readiness for the sport. The urban outdoor court setting adds to the dynamic and energetic atmosphere of the scene."
text2 = "In this image, a basketball player is shown making a super cool dunk in mid-air."
similarity_score1 = calculate_cosine_similarity(text1, text2)
print(f"Cosine Similarity Score with sklearn: {similarity_score1:.3f}")

similarity_score2 = spacy_similarity(text1, text2)
print(f"Semantic Similarity Score with spaCy: {similarity_score2:.3f}")

similarity_score3=calculate_cosine_similarity_with_embeddings(text1, text2)
print(f"Cosine Similarity Score with sentence transformer: {similarity_score3:.3f}")

similarity_score4 = 0.75
print(f"Cosine Similarity Score with human feedback: {similarity_score4:.3f}")

In [ ]:
def extract_rewritten_comment(response):
    """
    Extracts the rewritten comment from GPT-4o response.
    """
    lines = response.split('\n')
    rewritten_comment = []
    rewrite_started = False
    for line in lines:
        if "Rewritten Comment:" in line:
            rewrite_started = True
            continue
        if rewrite_started:
            if line.strip() == "":
                break
            rewritten_comment.append(line.strip())
    return " ".join(rewritten_comment)

### Examples

In [ ]:
import numpy as np
import sys
rscores =[]

scores=[]

### 1

In [ ]:
query_text = "Find a female soccer player that is playing."
import io
old_stdout = sys.stdout
new_stdout = io.StringIO()
sys.stdout = new_stdout
query_results = query_pinecone(query_text,1) # query, k
sys.stdout = old_stdout

collected_results = collect_query_results(query_results)

for result in collected_results:
  id= result['ID']
  score= result['Score']
  text= result['Text']
  frame= result['Frame Number']
  file_name= result['File Name']
  print(f"ID={id}")
  print(f"score={score}")
  print(f"text={text}")
  print(f"frame_number={frame}")
  print(f"file_name={file_name}")
  print()  # Add a newline for better readability between entries
response_content = get_response(text)
print(response_content)

In [ ]:
directory = "Chapter10/videos"
download(directory,file_name)
print("Displaying video: ",file_name)
display_video(file_name)

In [ ]:
text1 = "This image shows soccer players on a field dribbling and passing the ball."

text2 = extract_rewritten_comment(response_content)

print(f"Human Feedback Comment: {text1}")
print(f"Rewritten Comment: {text2}")

similarity_score3=calculate_cosine_similarity_with_embeddings(text1, text2)
print(f"Cosine Similarity Score with sentence transformer: {similarity_score3:.3f}")
scores.append(similarity_score3)
rscores.append(score)

### 2

In [ ]:
query_text = "Find a basketball player scoring with a slam dunk."
old_stdout = sys.stdout
new_stdout = io.StringIO()
sys.stdout = new_stdout
query_results = query_pinecone(query_text,1) # query, k
sys.stdout = old_stdout

collected_results = collect_query_results(query_results)

for result in collected_results:
  id= result['ID']
  score= result['Score']
  text= result['Text']
  frame= result['Frame Number']
  file_name= result['File Name']
  print(f"ID={id}")
  print(f"score={score}")
  print(f"text={text}")
  print(f"frame_number={frame}")
  print(f"file_name={file_name}")
  print()  # Add a newline for better readability between entries
response_content = get_response(text)
print(response_content)

In [ ]:
directory = "Chapter10/videos"
download(directory,file_name)
display_video(file_name)

In [ ]:
text1 = "This image shows an incredible dunk by a basketball player."

text2 = extract_rewritten_comment(response_content)

print(f"Human Feedback Comment: {text1}")
print(f"Rewritten Comment: {text2}")

similarity_score3=calculate_cosine_similarity_with_embeddings(text1, text2)
print(f"Cosine Similarity Score with sentence transformer: {similarity_score3:.3f}")
scores.append(similarity_score3)
rscores.append(score)

### Metrics calculation and display

In [ ]:
print(len(scores), scores)
print(len(rscores), rscores)

In [ ]:
mean_score = np.mean(scores)
median_score = np.median(scores)
std_deviation = np.std(scores)
variance = np.var(scores)
min_score = np.min(scores)
max_score = np.max(scores)
range_score = max_score - min_score
percentile_25 = np.percentile(scores, 25)
percentile_75 = np.percentile(scores, 75)
iqr = percentile_75 - percentile_25

print(f"Mean: {mean_score:.2f}")
print(f"Median: {median_score:.2f}")
print(f"Standard Deviation: {std_deviation:.2f}")
print(f"Variance: {variance:.2f}")
print(f"Minimum: {min_score:.2f}")
print(f"Maximum: {max_score:.2f}")
print(f"Range: {range_score:.2f}")
print(f"25th Percentile (Q1): {percentile_25:.2f}")
print(f"75th Percentile (Q3): {percentile_75:.2f}")
print(f"Interquartile Range (IQR): {iqr:.2f}")

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

scores = np.array(scores)
rscores = np.array(rscores)

assert len(scores) == len(rscores), "Length of scores and rscores must be equal"

threshold = 0.6

true_labels = (rscores > threshold).astype(int)
predicted_labels = (scores > threshold).astype(int)

f1 = f1_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels)
accuracy = accuracy_score(true_labels, predicted_labels)

print(f"F1 Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Accuracy: {accuracy:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

conf_matrix = confusion_matrix(true_labels, predicted_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Negative', 'Predicted Positive'], yticklabels=['Actual Negative', 'Actual Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()